In [0]:
%pip install lightgbm==4.3.0
dbutils.library.restartPython()

In [0]:
# =========================================
# 0) Imports
# =========================================
import os
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import lightgbm as lgb

import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature

# =========================================
# 1) CONFIG (EDIT THESE)
# =========================================
# Path to your LightGBM checkpoint txt.
# Recommended: store it in a Unity Catalog Volume:
#   /Volumes/<catalog>/<schema>/<volume>/models/lab_model.txt
CHECKPOINT_PATH = "/Volumes/final_project/default/files/lgb_model.txt"

# Unity Catalog registered model name: <catalog>.<schema>.<model_name>
UC_MODEL_NAME = "final_project.default.fare_lgbm_pyfunc"

# Optional: set registry to Unity Catalog
mlflow.set_registry_uri("databricks-uc")


# =========================================
# 2) Feature engineering (Gold.ipynb-matching)
# =========================================
EDA_FEATURES = [
    "hs_dist",
    "tms_drop_distance",
    "plz_drop_distance",
    "met_drop_distance",
    "hbk_drop_distance",
    "pickup_longitude",
    "nyc_drop_distance",
    "wtc_drop_distance",
    "dropoff_longitude",
    "sol_drop_distance",
    "ewr_drop_distance",
    "lga_drop_distance",
    "passenger_count",
    "dropoff_latitude",
    "pickup_latitude",
    "jfk_drop_distance",
    "hour",
]

# Exactly from Gold.ipynb: (lon, lat)
SIGHTS_LONLAT: Dict[str, Tuple[float, float]] = {
    "jfk": (-73.7781, 40.6413),
    "lga": (-73.8740, 40.7769),
    "ewr": (-74.1745, 40.6895),
    "met": (-73.9632, 40.7794),
    "wtc": (-74.0099, 40.7126),
    "sol": (-74.0445, 40.6892),
    "nyc": (-74.0063889, 40.7141667),
    "tms": (-73.9854406, 40.7581047),
    "plz": (-73.9750593, 40.7651662),
    "hbk": (-74.0282202, 40.7356908),
    # "brb": (-73.9741970, 40.5882819),  # exists in Gold, not used in EDA_FEATURES
}

R_KM = 6378.0  # Gold.ipynb uses 6378.0


def haversine_km_lonlat(
    lon1: np.ndarray,
    lat1: np.ndarray,
    lon2: np.ndarray,
    lat2: np.ndarray,
    R: float = R_KM,
) -> np.ndarray:
    """
    Gold.ipynb equivalent:
      a = sin(dlat/2)^2 + cos(lat1)*cos(lat2)*sin(dlon/2)^2
      c = 2*asin(sqrt(a))
      d = R*c
    """
    lon1r = np.radians(lon1)
    lat1r = np.radians(lat1)
    lon2r = np.radians(lon2)
    lat2r = np.radians(lat2)

    dlon = lon2r - lon1r
    dlat = lat2r - lat1r

    a = (np.sin(dlat / 2.0) ** 2) + (np.cos(lat1r) * np.cos(lat2r) * (np.sin(dlon / 2.0) ** 2))
    c = 2.0 * np.arcsin(np.sqrt(a))
    return R * c


def build_features(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Input raw_df must contain:
      pickup_datetime, pickup_longitude, pickup_latitude,
      dropoff_longitude, dropoff_latitude, passenger_count

    Output: DataFrame with columns exactly in EDA_FEATURES order.
    """
    df = raw_df.copy()

    required = [
        "pickup_datetime",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required input columns: {missing}")

    # numeric columns
    for c in ["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude", "passenger_count"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if df[["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"]].isna().any().any():
        bad = df[["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"]].isna().any()
        bad_cols = bad[bad].index.tolist()
        raise ValueError(f"Some coordinates are NaN after conversion: {bad_cols}")

    # hour from pickup_datetime
    # Accepts strings, timestamps, etc.
    dt = pd.to_datetime(df["pickup_datetime"], errors="coerce", utc=False)
    if dt.isna().any():
        raise ValueError("Some pickup_datetime values could not be parsed.")
    df["hour"] = dt.dt.hour.astype(int)

    # hs_dist: pickup -> dropoff
    df["hs_dist"] = haversine_km_lonlat(
        lon1=df["pickup_longitude"].to_numpy(dtype=float),
        lat1=df["pickup_latitude"].to_numpy(dtype=float),
        lon2=df["dropoff_longitude"].to_numpy(dtype=float),
        lat2=df["dropoff_latitude"].to_numpy(dtype=float),
    )

    # *_drop_distance: sight -> dropoff
    drop_lon = df["dropoff_longitude"].to_numpy(dtype=float)
    drop_lat = df["dropoff_latitude"].to_numpy(dtype=float)

    for name in ["tms", "plz", "met", "hbk", "nyc", "wtc", "sol", "ewr", "lga", "jfk"]:
        sight_lon, sight_lat = SIGHTS_LONLAT[name]
        df[f"{name}_drop_distance"] = haversine_km_lonlat(
            lon1=np.full_like(drop_lon, sight_lon, dtype=float),
            lat1=np.full_like(drop_lat, sight_lat, dtype=float),
            lon2=drop_lon,
            lat2=drop_lat,
        )

    X = df.reindex(columns=EDA_FEATURES)

    if X.isna().any().any():
        bad_cols = X.columns[X.isna().any()].tolist()
        raise ValueError(f"NaNs found in engineered features: {bad_cols}")

    return X


# =========================================
# 3) MLflow pyfunc model wrapper
# =========================================
class FareLGBMPyFunc(mlflow.pyfunc.PythonModel):
    """
    MLflow pyfunc model that:
      - loads LightGBM Booster from an artifact checkpoint (.txt)
      - transforms raw inputs -> EDA_FEATURES
      - returns predictions
    """

    def load_context(self, context):
        ckpt = context.artifacts["checkpoint"]
        self.bst = lgb.Booster(model_file=ckpt)

    def predict(self, context, model_input: pd.DataFrame) -> pd.Series:
        # Ensure input is DataFrame
        if not isinstance(model_input, pd.DataFrame):
            model_input = pd.DataFrame(model_input)

        X = build_features(model_input)
        preds = self.bst.predict(X)

        # return as a pandas Series (nice for MLflow serving / Spark)
        return pd.Series(preds, name="prediction")


# =========================================
# 4) Log + register the model to Unity Catalog
# =========================================
# Define a stable input example for schema / serving
input_example = pd.DataFrame([{
    "pickup_datetime": "2013-07-06 17:18:00",
    "pickup_longitude": -73.9822,
    "pickup_latitude": 40.7612,
    "dropoff_longitude": -73.9995,
    "dropoff_latitude": 40.7320,
    "passenger_count": 1,
}])

# Create a signature (recommended)
X_ex = build_features(input_example)
bst_tmp = lgb.Booster(model_file=CHECKPOINT_PATH)
y_ex = bst_tmp.predict(X_ex)
signature = infer_signature(input_example, pd.Series(y_ex, name="prediction"))

with mlflow.start_run() as run:
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=FareLGBMPyFunc(),
        artifacts={"checkpoint": CHECKPOINT_PATH},
        input_example=input_example,
        signature=signature,
        # These dependencies become the model environment used by Serving
        pip_requirements=[
            "lightgbm==4.3.0",
            "numpy",
            "pandas",
            "mlflow",
        ],
        registered_model_name=UC_MODEL_NAME,
    )

print("✅ Logged and registered UC model:", UC_MODEL_NAME)
print("Run ID:", run.info.run_id)


# =========================================
# 5) (Optional) Local smoke test via MLflow loaded model
# =========================================
# Load latest registered version by name (may require specifying version/stage in your workflow)
# For a quick test from the run artifact:
model_uri = f"runs:/{run.info.run_id}/model"
loaded = mlflow.pyfunc.load_model(model_uri)

test_df = pd.DataFrame([{
    "pickup_datetime": "2013-07-06 17:18:00",
    "pickup_longitude": -73.9822,
    "pickup_latitude": 40.7612,
    "dropoff_longitude": -73.9995,
    "dropoff_latitude": 40.7320,
    "passenger_count": 1,
}])

print("Prediction:", loaded.predict(test_df).iloc[0])
